# Notebook 1: ClinVar Data Preparation and Quality Control

## Objective
Download, filter, and preprocess ClinVar variant data focused on DNA repair genes. This notebook prepares a clean, analysis-ready dataset by:
1. Downloading ClinVar variant_summary.txt
2. Filtering to pathogenic and benign variants in DNA repair genes
3. Applying quality controls and removing duplicates
4. Exploring data structure and completeness
5. Saving processed data for downstream analysis



## Part 1: Import Libraries and Configure Environment

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Add scripts to path
sys.path.insert(0, os.path.abspath('../scripts'))

# Import custom functions
from data_loader import (
    download_clinvar,
    load_clinvar_raw,
    filter_to_dna_repair_genes,
    filter_clinical_significance,
    filter_review_status,
    remove_duplicates,
    select_columns
)

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully.
Pandas version: 2.2.3
NumPy version: 2.2.5


## Part 2: Download ClinVar Data

This section downloads the latest ClinVar variant_summary.txt file from NCBI. The file is ~200 MB compressed, ~700 MB uncompressed. If already downloaded, the step is skipped.

In [2]:
# Set paths
raw_data_dir = os.path.abspath('../data/raw')
processed_data_dir = os.path.abspath('../data/processed')
gz_path = os.path.join(raw_data_dir, 'variant_summary.txt.gz')
txt_path = os.path.join(raw_data_dir, 'variant_summary.txt')

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

print(f"Raw data directory: {raw_data_dir}")
print(f"Processed data directory: {processed_data_dir}")

# Check if file already exists
if os.path.exists(txt_path):
    print(f"\n✓ ClinVar file already downloaded: {txt_path}")
    print(f"File size: {os.path.getsize(txt_path) / (1024**2):.1f} MB")
else:
    print("\nDownloading ClinVar variant_summary.txt (this may take 2-5 minutes)...")
    txt_path = download_clinvar(gz_path)
    print(f"File size: {os.path.getsize(txt_path) / (1024**2):.1f} MB")

Raw data directory: c:\Users\hp\Documents\New project 2\data\raw
Processed data directory: c:\Users\hp\Documents\New project 2\data\processed

✓ ClinVar file already downloaded: c:\Users\hp\Documents\New project 2\data\raw\variant_summary.txt
File size: 3722.7 MB


## Part 3: Load and Explore Raw Data

In [3]:
# Load raw ClinVar data
df_raw = load_clinvar_raw(txt_path)

print(f"\n--- RAW DATA OVERVIEW ---")
print(f"Total rows: {len(df_raw):,}")
print(f"Total columns: {len(df_raw.columns)}")
print(f"\nColumn names:")
print(df_raw.columns.tolist())

Loading ClinVar data from c:\Users\hp\Documents\New project 2\data\raw\variant_summary.txt...
Loaded 8918806 total variants

--- RAW DATA OVERVIEW ---
Total rows: 8,918,806
Total columns: 7

Column names:
['Type', 'GeneSymbol', 'ClinicalSignificance', 'Chromosome', 'Start', 'Stop', 'ReviewStatus']


In [4]:
# Inspect data structure
print("\n--- DATA TYPES ---")
print(df_raw.dtypes)

print("\n--- FIRST FEW ROWS ---")
print(df_raw[['GeneSymbol', 'Type', 'ClinicalSignificance', 'Chromosome', 'Start', 'Stop']].head(10))


--- DATA TYPES ---
Type                    object
GeneSymbol              object
ClinicalSignificance    object
Chromosome              object
Start                    Int64
Stop                     Int64
ReviewStatus            object
dtype: object

--- FIRST FEW ROWS ---
  GeneSymbol                       Type          ClinicalSignificance  \
0      AP5Z1                      Indel  Pathogenic/Likely pathogenic   
1      AP5Z1                      Indel  Pathogenic/Likely pathogenic   
2      AP5Z1                   Deletion                    Pathogenic   
3      AP5Z1                   Deletion                    Pathogenic   
4     ZNF592  single nucleotide variant        Uncertain significance   
5     ZNF592  single nucleotide variant        Uncertain significance   
6    FOXRED1  single nucleotide variant                    Pathogenic   
7    FOXRED1  single nucleotide variant                    Pathogenic   
8    FOXRED1  single nucleotide variant             Likely pathogeni

In [5]:
# Check for missing values in key columns
key_cols = ['GeneSymbol', 'Type', 'ClinicalSignificance', 'Chromosome', 'Start', 'Stop']
print("\n--- MISSING VALUES IN KEY COLUMNS ---")
missing_summary = df_raw[key_cols].isnull().sum()
print(missing_summary)
print(f"\nPercent missing (by column):")
print((missing_summary / len(df_raw) * 100).round(2))


--- MISSING VALUES IN KEY COLUMNS ---
GeneSymbol              0
Type                    0
ClinicalSignificance    0
Chromosome              0
Start                   0
Stop                    0
dtype: int64

Percent missing (by column):
GeneSymbol              0.0
Type                    0.0
ClinicalSignificance    0.0
Chromosome              0.0
Start                   0.0
Stop                    0.0
dtype: float64


## Part 4: Filter to DNA Repair Genes

In [6]:
# Define gene list (19 DNA repair genes)
dna_repair_genes = [
    'MLH1', 'MSH2', 'MSH6', 'PMS2',  # MMR
    'BRCA1', 'BRCA2', 'RAD51', 'XRCC3',  # HR
    'APE1', 'XRCC1',  # BER
    'XPA', 'XPC', 'ERCC1',  # NER
    'TP53', 'CHEK2', 'ATM', 'PTEN',  # Checkpoint
    'POLE', 'POLD1'  # Polymerase
]

print(f"DNA Repair Genes (n={len(dna_repair_genes)}):")
print(", ".join(sorted(dna_repair_genes)))

# Filter
df_dna_repair = filter_to_dna_repair_genes(df_raw, gene_list=dna_repair_genes)
print(f"\nAfter filtering to {len(dna_repair_genes)} genes:")
print(f"  Variants retained: {len(df_dna_repair):,} ({len(df_dna_repair)/len(df_raw)*100:.1f}%)")

DNA Repair Genes (n=19):
APE1, ATM, BRCA1, BRCA2, CHEK2, ERCC1, MLH1, MSH2, MSH6, PMS2, POLD1, POLE, PTEN, RAD51, TP53, XPA, XPC, XRCC1, XRCC3
Filtered to 238523 variants in DNA repair genes

After filtering to 19 genes:
  Variants retained: 238,523 (2.7%)


## Part 5: Filter to Pathogenic and Benign Variants

In [7]:
# Check available clinical significance categories
print("Clinical significance categories in DNA repair subset:")
print(df_dna_repair['ClinicalSignificance'].value_counts())

# Filter to Pathogenic and Benign only
df_filtered = filter_clinical_significance(df_dna_repair)
print(f"\nAfter filtering to Pathogenic/Benign only:")
print(f"  Variants retained: {len(df_filtered):,}")

Clinical significance categories in DNA repair subset:
ClinicalSignificance
Uncertain significance                                       80905
Likely benign                                                49178
Pathogenic                                                   40532
Conflicting classifications of pathogenicity                 32529
Benign/Likely benign                                         11568
Benign                                                        7000
Likely pathogenic                                             6591
Pathogenic/Likely pathogenic                                  5093
-                                                             4852
not provided                                                   229
drug response                                                   15
no classification for the single variant                        13
association                                                      8
no classifications from unflagged records            

## Part 6: Apply Quality Filters

In [8]:
# Filter by review status
df_reviewed = filter_review_status(df_filtered)
print(f"After review status filter:")
print(f"  Variants retained: {len(df_reviewed):,}")

Filtered to 47532 variants with review status information
After review status filter:
  Variants retained: 47,532


In [9]:
# Remove duplicates
df_dedup = remove_duplicates(df_reviewed)
print(f"\nAfter deduplication:")
print(f"  Variants retained: {len(df_dedup):,}")

Removed 4678 duplicates, retained 42854 variants

After deduplication:
  Variants retained: 42,854


## Part 7: Select and Standardize Columns

In [10]:
# Select and rename key columns
df_processed = select_columns(df_dedup)

print("\n--- PROCESSED DATA STRUCTURE ---")
print(f"Columns: {df_processed.columns.tolist()}")
print(f"\nShape: {df_processed.shape}")
print(f"\nData types:")
print(df_processed.dtypes)


--- PROCESSED DATA STRUCTURE ---
Columns: ['Chromosome', 'Start', 'Stop', 'VariantType', 'Gene', 'ClinicalSignif', 'ReviewStatus']

Shape: (42854, 7)

Data types:
Chromosome        object
Start              Int64
Stop               Int64
VariantType       object
Gene              object
ClinicalSignif    object
ReviewStatus      object
dtype: object


In [11]:
# Preview processed data
print("\n--- SAMPLE OF PROCESSED DATA ---")
print(df_processed.head(10))


--- SAMPLE OF PROCESSED DATA ---
    Chromosome     Start      Stop                VariantType Gene  \
489         na        -1        -1                  Insertion  XPC   
491          3  14200090  14200091                   Deletion  XPC   
492          3  14158590  14158591                   Deletion  XPC   
493          3  14197833  14197833  single nucleotide variant  XPC   
494          3  14156333  14156333  single nucleotide variant  XPC   
495          3  14208723  14208724             Microsatellite  XPC   
496          3  14167223  14167224             Microsatellite  XPC   
497          3  14199648  14199648  single nucleotide variant  XPC   
498          3  14158148  14158148  single nucleotide variant  XPC   
499          3  14209889  14209889  single nucleotide variant  XPC   

    ClinicalSignif                                       ReviewStatus  
489     Pathogenic                     no assertion criteria provided  
491     Pathogenic                criteria provided

## Part 8: Explore Allele Frequency Data Availability

In [13]:
# Check allele frequency completeness
if 'AlleleFreq' in df_processed.columns:
    af_available = df_processed['AlleleFreq'].notna().sum()
    af_total = len(df_processed)
    af_coverage = af_available / af_total * 100
    
    print(f"\n--- ALLELE FREQUENCY DATA ---")
    print(f"Variants with AlleleFreq data: {af_available:,} / {af_total:,} ({af_coverage:.1f}%)")
    
    if af_available > 0:
        # Check by clinical significance
        print(f"\nAllele frequency coverage by clinical significance:")
        af_by_sig = df_processed.groupby('ClinicalSignif')['AlleleFreq'].apply(
            lambda x: f"{x.notna().sum()}/{len(x)} ({x.notna().sum()/len(x)*100:.1f}%)"
        )
        print(af_by_sig)
    else:
        print("\n⚠ WARNING: AlleleFreq field is empty. Allele frequency analysis will be omitted.")
else:
    print("\nNOTE: Allele frequency data not available in ClinVar dataset. Skipping frequency analysis.")


NOTE: Allele frequency data not available in ClinVar dataset. Skipping frequency analysis.


## Part 9: Generate Dataset Summary Statistics

In [14]:
print("\n" + "="*60)
print("PROCESSED DATASET SUMMARY")
print("="*60)

print(f"\nTotal variants: {len(df_processed):,}")
print(f"Total genes: {df_processed['Gene'].nunique()}")
print(f"Total chromosomes: {df_processed['Chromosome'].nunique()}")

print(f"\nClinical Significance:")
for sig in ['Pathogenic', 'Benign']:
    count = (df_processed['ClinicalSignif'] == sig).sum()
    pct = count / len(df_processed) * 100
    print(f"  {sig}: {count:,} ({pct:.1f}%)")

print(f"\nVariant Types:")
vtype_counts = df_processed['VariantType'].value_counts()
for vtype, count in vtype_counts.items():
    pct = count / len(df_processed) * 100
    print(f"  {vtype}: {count:,} ({pct:.1f}%)")

print(f"\nTop 10 Genes by Variant Count:")
top_genes = df_processed['Gene'].value_counts().head(10)
for gene, count in top_genes.items():
    pct = count / len(df_processed) * 100
    print(f"  {gene}: {count:,} ({pct:.1f}%)")


PROCESSED DATASET SUMMARY

Total variants: 42,854
Total genes: 55
Total chromosomes: 14

Clinical Significance:
  Pathogenic: 36,304 (84.7%)
  Benign: 6,550 (15.3%)

Variant Types:
  Deletion: 18,623 (43.5%)
  single nucleotide variant: 12,761 (29.8%)
  Duplication: 6,559 (15.3%)
  Insertion: 1,806 (4.2%)
  Indel: 1,670 (3.9%)
  Microsatellite: 1,371 (3.2%)
  copy number loss: 37 (0.1%)
  Inversion: 19 (0.0%)
  copy number gain: 5 (0.0%)
  Variation: 2 (0.0%)
  Complex: 1 (0.0%)

Top 10 Genes by Variant Count:
  BRCA2: 10,442 (24.4%)
  BRCA1: 8,148 (19.0%)
  ATM: 5,585 (13.0%)
  MSH2: 3,679 (8.6%)
  MSH6: 3,582 (8.4%)
  MLH1: 2,992 (7.0%)
  PTEN: 1,724 (4.0%)
  PMS2: 1,621 (3.8%)
  TP53: 1,552 (3.6%)
  CHEK2: 1,526 (3.6%)


## Part 10: Save Processed Data

In [15]:
# Save to CSV
output_path = os.path.join(processed_data_dir, 'variants_filtered.csv')
df_processed.to_csv(output_path, index=False)

print(f"✓ Processed data saved to: {output_path}")
print(f"  File size: {os.path.getsize(output_path) / (1024**2):.1f} MB")
print(f"  Rows: {len(df_processed):,}")
print(f"  Columns: {len(df_processed.columns)}")

✓ Processed data saved to: c:\Users\hp\Documents\New project 2\data\processed\variants_filtered.csv
  File size: 3.6 MB
  Rows: 42,854
  Columns: 7


## Part 11: Document Data Preparation Limitations

In [16]:
limitations = """
DATA PREPARATION LIMITATIONS AND NOTES
======================================

1. CURATED ANNOTATIONS ONLY
   This analysis relies exclusively on ClinVar curated annotations and does not
   incorporate population-scale frequency data from external sources (e.g., gnomAD)
   or experimental validation studies.

2. ALLELE FREQUENCY DATA
   Allele frequency information is conditional on availability in the ClinVar
   variant_summary.txt file. If sparse or missing, frequency-based comparisons
   will be omitted in downstream analysis.

3. VARIANT CLASSIFICATION
   Clinical significance classifications (Pathogenic/Benign) reflect ClinVar
   community consensus. This is a curated consensus, not an independent assessment.

4. GENE GENE FILTERING
   Variants were filtered to a specific set of 19 clinically important DNA repair
   genes. Other genes in the ClinVar database were excluded.

5. UNCERTAIN SIGNIFICANCE EXCLUDED
   Variants classified as "Uncertain significance" were excluded to enable clear
   binary comparison between pathogenic and benign variants.

6. NO EXPERIMENTAL VALIDATION
   This analysis does not include independent functional studies or experimental
   validation of pathogenicity predictions.

7. TRANSCRIPT ISOFORM MAPPING
   Variants are mapped to genes, not specific isoforms or protein domains.
   Transcript-level annotation is treated as an optional extension.

"""

print(limitations)


DATA PREPARATION LIMITATIONS AND NOTES

1. CURATED ANNOTATIONS ONLY
   This analysis relies exclusively on ClinVar curated annotations and does not
   incorporate population-scale frequency data from external sources (e.g., gnomAD)
   or experimental validation studies.

2. ALLELE FREQUENCY DATA
   Allele frequency information is conditional on availability in the ClinVar
   variant_summary.txt file. If sparse or missing, frequency-based comparisons
   will be omitted in downstream analysis.

3. VARIANT CLASSIFICATION
   Clinical significance classifications (Pathogenic/Benign) reflect ClinVar
   community consensus. This is a curated consensus, not an independent assessment.

4. GENE GENE FILTERING
   Variants were filtered to a specific set of 19 clinically important DNA repair
   genes. Other genes in the ClinVar database were excluded.

5. UNCERTAIN SIGNIFICANCE EXCLUDED
   Variants classified as "Uncertain significance" were excluded to enable clear
   binary comparison between p

## Summary

✓ ClinVar data downloaded and preprocessed successfully
✓ Filtered to 19 DNA repair genes
✓ Retained pathogenic and benign variants only
✓ Applied quality controls and removed duplicates
✓ {f"{len(df_processed):,}"} variants ready for analysis
✓ Data saved to `data/processed/variants_filtered.csv`

**Next step:** Open and run Notebook 02 for analysis and interpretation.